# RetailFlow Source Profiling

Exploratory profiling of RetailFlow's raw incoming source data, ahead of building Bronze ingestion (RF-002).


## Objective

Explore the shape and quality of the raw `customers`, `orders`, and `payments` source data before writing Bronze ingestion or Silver cleaning logic. This notebook is exploration only — it informs the engineering decisions that Bronze and Silver will implement.

> **Note**
>
> - **Development** Parquet data will be used for exploration (`data/incoming/development/`).
> - Incoming records are **intentionally dirty** — see `docs/data_contracts.md` for the full list of injected data-quality issue codes and rates.
> - This notebook **must not modify source files**. Treat everything under `data/incoming/` as read-only.
> - **Bronze** will preserve raw values exactly, including invalid ones.
> - **Cleaning and quarantine belong to Silver** — not to this notebook, and not to Bronze.


## Tools Used

- **pandas** / **pyarrow** — reading and inspecting the Parquet source files
- **pytest** (separately, in `tests/`) — for any profiling helpers that graduate out of this notebook

_(This list documents intent only. Profiling logic is developed incrementally below. Reusable logic will later move into src/retailflow/profiling.py.)_


## Source Configuration

Point at the `development` profile under `data/incoming/development/` (see `config/data_generation.yml` and `data/incoming/development/manifest.json` for row counts, file layout, and the injected issue catalogue). Do not point this notebook at `portfolio` for routine exploration — it is much larger and intended for cloud-scale demonstrations.


In [1]:
from pathlib import Path

import pandas as pd

print("pandas version:", pd.__version__)


pandas version: 2.2.2


In [2]:
working_dir = Path.cwd()

if working_dir.name == "notebooks":
    project_root = working_dir.parent
else:
    project_root = working_dir

# Print paths relative to the project root only - never the machine-specific
# absolute path - so notebook outputs stay portable across machines.
print("Working directory (relative to project root):", working_dir.relative_to(project_root))

# Use the development profile: small enough to load quickly for exploration;
# portfolio is reserved for cloud-scale demonstrations (see docs/data_contracts.md).
source_root = project_root / "data" / "incoming" / "development"
customers = pd.read_parquet(source_root / "customers")
orders = pd.read_parquet(source_root / "orders")
payments = pd.read_parquet(source_root / "payments")

# Confirm the loaded row counts match data/incoming/development/manifest.json
# before profiling further.
print("Customer data shape:", customers.shape)
print("Orders data shape:", orders.shape)
print("Payments data shape:", payments.shape)


Working directory (relative to project root): notebooks


Customer data shape: (10000, 6)
Orders data shape: (100000, 7)
Payments data shape: (130000, 8)


## Customer Profiling

Explore `customers`: row counts, null/blank rates, duplicate `customer_id` values, invalid `province`/`customer_status` domain values, whitespace/casing anomalies in `full_name`, and malformed emails. Compare against the issue codes documented in `docs/data_contracts.md`.


### Customer overview

Sample the data, list columns, and check overall null counts before deeper checks.


In [3]:
display(customers.head())

print("Customer columns:")
print(customers.columns.tolist())

print("\nActual NULL values:")
print(customers.isna().sum())


,customer_id,full_name,email,province,signup_date,customer_status
0,CUST-000001,Danielle Gould,danielle.gould1@example.com,NL,2025-11-26,active
1,CUST-000002,Angel Strong,angel.strong2@example.net,MB,2023-11-25,active
2,CUST-000003,Joshua Warner,joshua.warner3@example.org,NT,2025-05-18,inactive
3,CUST-000004,Jeffrey Cannon,jeffrey.cannon4@example.org,ON,2023-12-05,active
4,CUST-000005,Jill Johnson,jill.johnson5@example.org,NB,2024-05-10,active


Customer columns:
['customer_id', 'full_name', 'email', 'province', 'signup_date', 'customer_status']

Actual NULL values:
customer_id        5
full_name          0
email              0
province           0
signup_date        0
customer_status    0
dtype: int64


### Missing and blank values

Count true nulls separately from blank/whitespace-only strings for every column — a null and a blank are different data-quality problems.


In [4]:
customer_quality = []

for col in customers.columns:
    series = customers[col]
    null_count = int(series.isna().sum())
    # Cast to pandas' nullable "string" dtype (not the Python str type) so
    # missing values stay NaN instead of being stringified as "nan"/"None"
    # before the blank check below.
    blank_count = int(
        (
            series.notna()
            & series.astype("string").str.strip().eq("")
        ).sum()
    )
    customer_quality.append(
        {
            "column": col,
            "null_count": null_count,
            "blank_count": blank_count,
        }
    )

customer_quality_df = pd.DataFrame(customer_quality)
display(customer_quality_df)


,column,null_count,blank_count
0,customer_id,5,3
1,full_name,0,0
2,email,0,106
3,province,0,0
4,signup_date,0,0
5,customer_status,0,11


### Customer ID completeness

Check `customer_id` specifically for null or blank values — a primary key must be complete.


In [5]:
customer_ids = customers["customer_id"]

null_mask = customer_ids.isna()
blank_mask = customer_ids.notna() & customer_ids.astype("string").str.strip().eq("")

total_rows = len(customers)
missing_id_count = int((null_mask | blank_mask).sum())
valid_id_count = total_rows - missing_id_count
completeness_rate = valid_id_count / total_rows

print("Total rows:", total_rows)
print("Null customer IDs:", int(null_mask.sum()))
print("Blank customer IDs:", int(blank_mask.sum()))
print("Missing customer IDs:", missing_id_count)
print("Valid customer IDs:", valid_id_count)
print(
    f"Customer ID completeness (valid_id_count / {total_rows} total rows): "
    f"{completeness_rate:.2%}"
)


Total rows: 10000
Null customer IDs: 5
Blank customer IDs: 3
Missing customer IDs: 8
Valid customer IDs: 9992
Customer ID completeness (valid_id_count / 10000 total rows): 99.92%


### Duplicate customer IDs

Check for duplicate `customer_id` values among rows that have a complete ID (missing IDs are already counted separately above) — a primary key must be unique.


In [6]:
valid_customer_ids = customer_ids[~(null_mask | blank_mask)]

duplicate_mask = valid_customer_ids.duplicated(keep=False)
duplicated_id_counts = valid_customer_ids[duplicate_mask].value_counts()

duplicate_row_count = int(duplicate_mask.sum())
distinct_duplicated_id_count = len(duplicated_id_counts)
excess_duplicate_row_count = duplicate_row_count - distinct_duplicated_id_count
duplicate_row_rate = duplicate_row_count / total_rows

print("Distinct duplicated customer IDs:", distinct_duplicated_id_count)
print("Rows containing duplicated customer IDs:", duplicate_row_count)
print("Excess duplicate rows:", excess_duplicate_row_count)
print(
    f"Duplicate-row rate (rows containing a duplicated ID / {total_rows} total rows): "
    f"{duplicate_row_rate:.2%}"
)

display(duplicated_id_counts.head(10))


Distinct duplicated customer IDs: 20
Rows containing duplicated customer IDs: 40
Excess duplicate rows: 20
Duplicate-row rate (rows containing a duplicated ID / 10000 total rows): 0.40%


customer_id
CUST-000205    2
CUST-001091    2
CUST-009856    2
CUST-008477    2
CUST-007904    2
CUST-007762    2
CUST-006943    2
CUST-006659    2
CUST-006546    2
CUST-005805    2
Name: count, dtype: int64

In [7]:
duplicated_customers = (
    customers[customers["customer_id"].isin(duplicated_id_counts.index)]
    .sort_values("customer_id")
)

print("Duplicated records:", len(duplicated_customers))
display(duplicated_customers.head(20))


Duplicated records: 40


,customer_id,full_name,email,province,signup_date,customer_status
204,CUST-000205,Todd Gomez,todd.gomez205@example.org,ON,2024-10-09,active
205,CUST-000205,Jason Higgins,jason.higgins206@example.com,AB,2023-04-09,inactive
1090,CUST-001091,Sandra Hill,sandra.hill1091@example.org,NU,2023-08-13,inactive
1091,CUST-001091,Carlos Bonilla,carlos.bonilla1092@example.org,XX,2023-07-18,active
1334,CUST-001335,Erica Cox,erica.cox1335@example.org,NU,2023-06-07,inactive
1335,CUST-001335,John Carter,john.carter1336@example.net,SK,2025-11-30,inactive
2681,CUST-002682,Gary Flores,gary.flores2682@example.com,SK,2024-03-19,inactive
2682,CUST-002682,David Hicks,david.hicks2683@example.org,YT,2025-03-15,inactive
2795,CUST-002795,sTEVEN cHRISTENSEN,steven.christensen2796@example.net,MB,2023-11-30,inactive
2794,CUST-002795,Frank Thomas,frank.thomas2795@example.com,NT,2025-03-13,active


In [8]:
customers[
    customers["customer_id"].eq("CUST-000205")
]


,customer_id,full_name,email,province,signup_date,customer_status
204,CUST-000205,Todd Gomez,todd.gomez205@example.org,ON,2024-10-09,active
205,CUST-000205,Jason Higgins,jason.higgins206@example.com,AB,2023-04-09,inactive


#### Finding

`CUST-000205` is a confirmed **conflicting** duplicate: the same customer ID is assigned to two records with different names, emails, provinces, signup dates, and statuses (Todd Gomez vs. Jason Higgins). This is one manually confirmed example, not a claim about all 20 distinct duplicated IDs — most duplicates have not been individually inspected, and some may be exact/harmless duplicates rather than conflicting ones. See Customer engineering decisions below for how this is handled.


### Province validation

Check `province` against the Canadian province/territory codes documented in `docs/data_contracts.md`. `province` has no missing-value issue in the generator (confirmed 0 nulls above), so only invalid-domain values are checked here.


In [9]:
valid_provinces = {
    "AB", "BC", "MB", "NB", "NL", "NS", "NT", "NU", "ON", "PE", "QC", "SK", "YT",
}

province_series = customers["province"]
invalid_province_mask = province_series.notna() & ~province_series.isin(valid_provinces)
invalid_province_count = int(invalid_province_mask.sum())
invalid_province_rate = invalid_province_count / total_rows

print(
    f"Invalid province codes (of {total_rows} total rows): "
    f"{invalid_province_count} ({invalid_province_rate:.2%})"
)
display(province_series[invalid_province_mask].value_counts().head(10))


Invalid province codes (of 10000 total rows): 70 (0.70%)


province
XX        18
on        16
Quebec    13
ZZ        13
QQ        10
Name: count, dtype: int64

### Customer-status validation

Check `customer_status` against the allowed domain (`active`, `inactive`) documented in `docs/data_contracts.md`.


In [10]:
valid_customer_statuses = {"active", "inactive"}

status_series = customers["customer_status"]
invalid_status_mask = status_series.notna() & ~status_series.isin(valid_customer_statuses)
invalid_status_count = int(invalid_status_mask.sum())
invalid_status_rate = invalid_status_count / total_rows

print(
    f"Invalid customer_status values (of {total_rows} total rows): "
    f"{invalid_status_count} ({invalid_status_rate:.2%})"
)
display(status_series[invalid_status_mask].value_counts().head(10))


Invalid customer_status values (of 10000 total rows): 58 (0.58%)


customer_status
ACTIVE       15
unknown      12
             11
suspended    10
Active       10
Name: count, dtype: int64

### Full-name validation

`docs/data_contracts.md` does not define a strict format for `full_name`; it only documents two injected issues, whitespace padding and casing. Whitespace padding is checked exactly (comparing to a stripped copy). Casing is checked with a **heuristic** — comparing to Python's title-case reconstruction — since there is no stricter rule in the data contract to validate against.


In [11]:
full_name_series = customers["full_name"]
stripped_full_names = full_name_series.str.strip()

whitespace_mask = full_name_series.notna() & full_name_series.ne(stripped_full_names)
whitespace_count = int(whitespace_mask.sum())
whitespace_rate = whitespace_count / total_rows

# Heuristic only: the injected MIXED_CASING issue swaps case (e.g. "jOHN sMITH").
# Comparing to a title-case reconstruction is a reasonable proxy given the data
# contract does not define a strict name-casing rule.
mixed_casing_mask = (
    full_name_series.notna()
    & stripped_full_names.ne("")
    & stripped_full_names.ne(stripped_full_names.str.title())
)
mixed_casing_count = int(mixed_casing_mask.sum())
mixed_casing_rate = mixed_casing_count / total_rows

print(
    f"Whitespace-padded names (of {total_rows} total rows): "
    f"{whitespace_count} ({whitespace_rate:.2%})"
)
print(
    f"Mixed-casing names, heuristic (of {total_rows} total rows): "
    f"{mixed_casing_count} ({mixed_casing_rate:.2%})"
)

overlap_count = int((whitespace_mask & mixed_casing_mask).sum())
print("Rows flagged by both checks (expected 0 - issues are mutually exclusive by design):", overlap_count)


Whitespace-padded names (of 10000 total rows): 82 (0.82%)
Mixed-casing names, heuristic (of 10000 total rows): 99 (0.99%)
Rows flagged by both checks (expected 0 - issues are mutually exclusive by design): 0


### Email validation

Check `email` for blanks and for malformed values. `docs/data_contracts.md` describes `MALFORMED_EMAIL` as the `@` character being replaced, so a populated email with no `@` at all is the deterministic signal for this issue — no stricter RFC-style email rule is applied, since the contract does not define one.


In [12]:
email_series = customers["email"]

email_blank_mask = email_series.notna() & email_series.str.strip().eq("")
email_blank_count = int(email_blank_mask.sum())
email_blank_rate = email_blank_count / total_rows

email_malformed_mask = (
    email_series.notna()
    & ~email_blank_mask
    & ~email_series.str.contains("@", regex=False)
)
email_malformed_count = int(email_malformed_mask.sum())
email_malformed_rate = email_malformed_count / total_rows

print(
    f"Blank emails (of {total_rows} total rows): "
    f"{email_blank_count} ({email_blank_rate:.2%})"
)
print(
    f"Malformed emails, missing '@' (of {total_rows} total rows): "
    f"{email_malformed_count} ({email_malformed_rate:.2%})"
)


Blank emails (of 10000 total rows): 106 (1.06%)
Malformed emails, missing '@' (of 10000 total rows): 92 (0.92%)


### Signup-date validation

Deterministic date logic: parse `signup_date` strictly as ISO-8601 (`YYYY-MM-DD`); anything that fails to parse is **invalid**. Anything that parses but falls after the profile's `as_of_date` (loaded from `data/incoming/development/manifest.json`, not hardcoded) is **future**.


In [13]:
import json

manifest_path = source_root / "manifest.json"
with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

as_of_date = pd.Timestamp(manifest["as_of_date"])
print("As-of date (from manifest.json):", as_of_date.date())

signup_date_series = customers["signup_date"]
parsed_signup_dates = pd.to_datetime(signup_date_series, format="%Y-%m-%d", errors="coerce")

invalid_signup_date_mask = signup_date_series.notna() & parsed_signup_dates.isna()
future_signup_date_mask = parsed_signup_dates.notna() & (parsed_signup_dates > as_of_date)

invalid_signup_date_count = int(invalid_signup_date_mask.sum())
future_signup_date_count = int(future_signup_date_mask.sum())
invalid_signup_date_rate = invalid_signup_date_count / total_rows
future_signup_date_rate = future_signup_date_count / total_rows

print(
    f"Invalid (unparseable) signup dates (of {total_rows} total rows): "
    f"{invalid_signup_date_count} ({invalid_signup_date_rate:.2%})"
)
print(
    f"Future signup dates, after {as_of_date.date()} (of {total_rows} total rows): "
    f"{future_signup_date_count} ({future_signup_date_rate:.2%})"
)


As-of date (from manifest.json): 2026-01-31
Invalid (unparseable) signup dates (of 10000 total rows): 37 (0.37%)
Future signup dates, after 2026-01-31 (of 10000 total rows): 26 (0.26%)


### Customer findings

Consolidate every check above into one summary table, then compare observed counts against the documented issue catalogue in `data/incoming/development/manifest.json`.


In [14]:
customer_quality_summary = pd.DataFrame([
    {
        "rule": "Missing customer_id (null or blank)",
        "failed_count": missing_id_count,
        "failure_rate": missing_id_count / total_rows,
        "treatment": "Quarantine in Silver (primary key required)",
    },
    {
        "rule": "Duplicate customer_id (rows sharing an ID)",
        "failed_count": duplicate_row_count,
        "failure_rate": duplicate_row_rate,
        "treatment": "Quarantine confirmed conflicts pending resolution rule",
    },
    {
        "rule": "Blank email",
        "failed_count": email_blank_count,
        "failure_rate": email_blank_rate,
        "treatment": "Preserve in Bronze; flag in Silver",
    },
    {
        "rule": "Malformed email (missing '@')",
        "failed_count": email_malformed_count,
        "failure_rate": email_malformed_rate,
        "treatment": "Preserve in Bronze; flag in Silver",
    },
    {
        "rule": "Invalid province code",
        "failed_count": invalid_province_count,
        "failure_rate": invalid_province_rate,
        "treatment": "Flag/quarantine per business criticality",
    },
    {
        "rule": "Invalid customer_status",
        "failed_count": invalid_status_count,
        "failure_rate": invalid_status_rate,
        "treatment": "Flag/quarantine per business criticality",
    },
    {
        "rule": "Invalid (unparseable) signup_date",
        "failed_count": invalid_signup_date_count,
        "failure_rate": invalid_signup_date_rate,
        "treatment": "Flag/quarantine per business criticality",
    },
    {
        "rule": "Future signup_date",
        "failed_count": future_signup_date_count,
        "failure_rate": future_signup_date_rate,
        "treatment": "Flag/quarantine per business criticality",
    },
    {
        "rule": "full_name whitespace padding",
        "failed_count": whitespace_count,
        "failure_rate": whitespace_rate,
        "treatment": "Repairable; standardize deterministically in Silver",
    },
    {
        "rule": "full_name mixed casing (heuristic)",
        "failed_count": mixed_casing_count,
        "failure_rate": mixed_casing_rate,
        "treatment": "Repairable; standardize deterministically in Silver",
    },
])

display(customer_quality_summary)


,rule,failed_count,failure_rate,treatment
0,Missing customer_id (null or blank),8,0.0008,Quarantine in Silver (primary key required)
1,Duplicate customer_id (rows sharing an ID),40,0.0040,Quarantine confirmed conflicts pending resolut...
2,Blank email,106,0.0106,Preserve in Bronze; flag in Silver
3,Malformed email (missing '@'),92,0.0092,Preserve in Bronze; flag in Silver
4,Invalid province code,70,0.0070,Flag/quarantine per business criticality
5,Invalid customer_status,58,0.0058,Flag/quarantine per business criticality
6,Invalid (unparseable) signup_date,37,0.0037,Flag/quarantine per business criticality
7,Future signup_date,26,0.0026,Flag/quarantine per business criticality
8,full_name whitespace padding,82,0.0082,Repairable; standardize deterministically in S...
9,full_name mixed casing (heuristic),99,0.0099,Repairable; standardize deterministically in S...


In [15]:
manifest_issue_counts = {
    issue["code"]: issue["affected_row_count"]
    for issue in manifest["issues"]["customers"]
}

observed_vs_documented = pd.DataFrame([
    {"code": "NULL_OR_BLANK_CUSTOMER_ID", "observed": missing_id_count},
    {"code": "DUPLICATE_CUSTOMER_ID", "observed": distinct_duplicated_id_count},
    {"code": "BLANK_EMAIL", "observed": email_blank_count},
    {"code": "MALFORMED_EMAIL", "observed": email_malformed_count},
    {"code": "INVALID_PROVINCE_CODE", "observed": invalid_province_count},
    {"code": "INVALID_CUSTOMER_STATUS", "observed": invalid_status_count},
    {"code": "INVALID_SIGNUP_DATE", "observed": invalid_signup_date_count},
    {"code": "FUTURE_SIGNUP_DATE", "observed": future_signup_date_count},
    {"code": "WHITESPACE_PADDING", "observed": whitespace_count},
    {"code": "MIXED_CASING", "observed": mixed_casing_count},
])
observed_vs_documented["documented"] = observed_vs_documented["code"].map(manifest_issue_counts)
observed_vs_documented["match"] = observed_vs_documented["observed"].eq(observed_vs_documented["documented"])

display(observed_vs_documented)


,code,observed,documented,match
0,NULL_OR_BLANK_CUSTOMER_ID,8,8,True
1,DUPLICATE_CUSTOMER_ID,20,20,True
2,BLANK_EMAIL,106,106,True
3,MALFORMED_EMAIL,92,92,True
4,INVALID_PROVINCE_CODE,70,70,True
5,INVALID_CUSTOMER_STATUS,58,58,True
6,INVALID_SIGNUP_DATE,37,37,True
7,FUTURE_SIGNUP_DATE,26,26,True
8,WHITESPACE_PADDING,82,82,True
9,MIXED_CASING,99,99,True


All observed counts match the documented issue catalogue exactly.

**Note on `DUPLICATE_CUSTOMER_ID`:** the manifest's `affected_row_count` (20) counts the rows whose ID was *overwritten* to create a duplicate — i.e. the number of distinct customer IDs that ended up duplicated (`distinct_duplicated_id_count`). Each of those 20 events produces a pair of rows sharing one ID, so the total row-level impact is 40 rows (`duplicate_row_count`), of which 20 are "excess" beyond the original single row. Both figures are correct; they answer different questions (distinct IDs affected vs. total rows involved), which is why `customer_quality_summary` above reports `duplicate_row_count` (40) while this comparison reports `distinct_duplicated_id_count` (20) to match the manifest's definition.


### Customer engineering decisions

- **Bronze preserves incoming values unchanged** — including missing, duplicate, and malformed customer records. No cleaning happens before Bronze.
- **Missing customer IDs** (null or blank, 8 rows / 0.08% of 10,000) are quarantined in Silver — a primary key must be present.
- **Duplicate customer IDs**: exact-duplicate vs. conflicting-duplicate classification is deferred to Silver, to be implemented using record hashes and Bronze lineage metadata — not by today's profiling.
- **Confirmed conflicting duplicates** (e.g. `CUST-000205` — Todd Gomez vs. Jason Higgins, differing on every other field) are quarantined pending an approved resolution rule. Only this one pair was manually confirmed as conflicting; this notebook does not claim all 20 distinct duplicated IDs are conflicting.
- **Repairable whitespace/casing problems** in `full_name` are standardized deterministically in Silver (e.g. `.str.strip()`, consistent title casing).
- **Malformed and blank emails** remain preserved in Bronze and are flagged (not repaired) in Silver — there is no reliable automatic fix for a broken email address.
- **Invalid `province` / `customer_status` / `signup_date` values** are flagged or quarantined in Silver according to business criticality, not automatically repaired here.
- **Profiling never modifies source Parquet files** — every check above reads `customers` read-only; no `.to_parquet()` call or in-place mutation occurs anywhere in this notebook.


## Order Profiling

Explore `orders`: row counts, null/blank/duplicate `order_id` values, orphaned or missing `customer_id` foreign keys, invalid or future timestamps, non-numeric/negative/blank amounts, and invalid status/channel/currency domain values.


## Payment Profiling

Explore `payments`: row counts, null/blank/duplicate `payment_id` values, orphaned or missing `order_id` foreign keys, invalid timestamps or payments recorded before their order, non-numeric/negative/blank/mismatched amounts, invalid method/status/currency values, and `failure_reason` consistency against `payment_status`.


## Data-Quality Findings

Summarize what profiling actually turned up: which issue codes were observed, roughly how often, and anything unexpected relative to `docs/data_contracts.md`.


## Engineering Decisions

Document the decisions these findings drive for Bronze and Silver — e.g. which columns need `TRY_CAST`/`TRY_TO_*` handling in Silver, what should be quarantined vs. repaired, and any open questions to resolve before implementing RF-002/RF-003.


## Interview Notes

Practice explaining this profiling work out loud, without notes: what data went in, what you looked for and why, what you found, and what you'd do differently — per the input/process/output/failure structure in `LEARNING_PLAN.md`.
